In [1]:
import os
import glob
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

In [2]:
# ========== 路径配置 ==========
BASE = "/kaggle/input/competitions/2026task2"
TRAIN_DIR = f"{BASE}/fer_data/fer_data/train"
TEST_DIR  = f"{BASE}/fer_data/fer_data/test"
TEMPLATE  = f"{BASE}/submission.csv"          # 模板


In [3]:
classes = ["Angry", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
class_to_idx = {c: i for i, c in enumerate(classes)}
print("类别映射：", class_to_idx)

# 检查文件夹是否存在
for c in classes:
    folder = os.path.join(TRAIN_DIR, c)
    print(c, "->", os.path.isdir(folder), "| 数量:", len(glob.glob(os.path.join(folder, "*"))))

类别映射： {'Angry': 0, 'Fear': 1, 'Happy': 2, 'Sad': 3, 'Surprise': 4, 'Neutral': 5}
Angry -> True | 数量: 3963
Fear -> True | 数量: 4097
Happy -> True | 数量: 7192
Sad -> True | 数量: 4862
Surprise -> True | 数量: 3202
Neutral -> True | 数量: 4959


In [4]:
# ========== 检查图片尺寸/通道 ==========
sample_imgs = glob.glob(os.path.join(TRAIN_DIR, classes[0], "*"))[:5]
for p in sample_imgs:
    img = Image.open(p)
    print(p, img.size, img.mode)

/kaggle/input/competitions/2026task2/fer_data/fer_data/train/Angry/bknqheuifa.jpg (48, 48) L
/kaggle/input/competitions/2026task2/fer_data/fer_data/train/Angry/ttawybtfpj.jpg (48, 48) L
/kaggle/input/competitions/2026task2/fer_data/fer_data/train/Angry/rvqtjtqkrm.jpg (48, 48) L
/kaggle/input/competitions/2026task2/fer_data/fer_data/train/Angry/pupguokcqt.jpg (48, 48) L
/kaggle/input/competitions/2026task2/fer_data/fer_data/train/Angry/nzezygfxpf.jpg (48, 48) L


In [5]:
# ========== Dataset ==========
class FERDataset(Dataset):
    def __init__(self, paths, labels=None, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, self.labels[idx]
        return img, self.paths[idx]

In [6]:
# ========== 收集训练数据 ==========
class_to_idx = {c: i for i, c in enumerate(classes)}
print("类别映射：", class_to_idx)

train_paths, train_labels = [], []
for c in classes:
    for p in glob.glob(os.path.join(TRAIN_DIR, c, "*")):
        train_paths.append(p)
        train_labels.append(class_to_idx[c])

print("总训练样本：", len(train_paths))
print("标签分布：", Counter(train_labels))

类别映射： {'Angry': 0, 'Fear': 1, 'Happy': 2, 'Sad': 3, 'Surprise': 4, 'Neutral': 5}
总训练样本： 28275
标签分布： Counter({2: 7192, 5: 4959, 3: 4862, 1: 4097, 0: 3963, 4: 3202})


In [7]:
# ========== 数据增强 ==========
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

In [8]:
# ========== 分层划分 ==========
X_tr, X_val, y_tr, y_val = train_test_split(
    train_paths, train_labels,
    test_size=0.15,
    stratify=train_labels,
    random_state=42
)

train_ds = FERDataset(X_tr, y_tr, transform=train_tfms)
val_ds   = FERDataset(X_val, y_val, transform=val_tfms)

# 类别不平衡：加权采样
counts = Counter(y_tr)
weights = [1.0 / counts[y] for y in y_tr]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

In [9]:
# ========== 模型 ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备：", device)

# resnet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

# eb0
#model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
#model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
#model = model.to(device)   # ← 加这一行

# resnet50
#model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
#model.fc = nn.Linear(model.fc.in_features, len(classes))
#model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

设备： cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


In [10]:
# ========== 训练/验证函数 ==========
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc="Train", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, desc="Val", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total, all_preds, all_labels

In [11]:
# ========== 训练 ==========
EPOCHS = 15
best_acc = 0
best_state = None

epoch_bar = tqdm(range(EPOCHS), desc="Epochs")
for epoch in epoch_bar:
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    epoch_bar.set_postfix(
        tr_loss=f"{tr_loss:.4f}",
        tr_acc=f"{tr_acc:.4f}",
        val_loss=f"{val_loss:.4f}",
        val_acc=f"{val_acc:.4f}",
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print("最佳验证准确率：", best_acc)
model.load_state_dict(best_state)

Epochs:   0%|          | 0/15 [00:00<?, ?it/s]

Train:   0%|          | 0/752 [00:00<?, ?it/s]

Val:   0%|          | 0/133 [00:00<?, ?it/s]

Train:   0%|          | 0/752 [00:00<?, ?it/s]

Val:   0%|          | 0/133 [00:00<?, ?it/s]

Train:   0%|          | 0/752 [00:00<?, ?it/s]

... (23 outputs left out)

Train:   0%|          | 0/752 [00:00<?, ?it/s]

Val:   0%|          | 0/133 [00:00<?, ?it/s]

最佳验证准确率： 0.6763319189061763


<All keys matched successfully>

In [12]:
# ========== 测试集预测 ==========
template = pd.read_csv(TEMPLATE)
test_ids = template["ID"].tolist()
print("模板行数：", len(test_ids))
print(template.head())

# 直接扫描测试目录（不再依赖 submission..txt）
test_paths_all = glob.glob(os.path.join(TEST_DIR, "*.jpg"))
print("测试图片数量：", len(test_paths_all))
print("测试文件名示例：", [os.path.basename(p) for p in test_paths_all[:5]])

# 建立 文件名 -> 完整路径 映射
id_to_path = {os.path.basename(p): p for p in test_paths_all}

# 检查模板 ID 和测试文件是否对得上
missing = [i for i in test_ids if i not in id_to_path]
print("模板里有但测试目录找不到的数量：", len(missing))
print("缺失示例：", missing[:10])
print("模板 ID 示例：", test_ids[:5])
print("测试文件名示例：", list(id_to_path.keys())[:5])

# 按模板 ID 顺序取路径
test_paths_ordered = [id_to_path[i] for i in test_ids]

test_ds = FERDataset(test_paths_ordered, labels=None, transform=val_tfms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds.extend(outputs.argmax(1).cpu().numpy())
    return preds

pred_labels = predict(model, test_loader)
print("预测数量：", len(pred_labels))

模板行数： 7065
               ID  Emotion
0  tborkgneql.jpg        0
1  ocnyeeiise.jpg        0
2  xxzktmwpao.jpg        0
3  qaheuveetu.jpg        0
4  xmtolvgkaf.jpg        0
测试图片数量： 7065
测试文件名示例： ['vlvyvlvywu.jpg', 'xzhzduujjb.jpg', 'mwuenemtet.jpg', 'bkckzwkzde.jpg', 'doybvlwefm.jpg']
模板里有但测试目录找不到的数量： 0
缺失示例： []
模板 ID 示例： ['tborkgneql.jpg', 'ocnyeeiise.jpg', 'xxzktmwpao.jpg', 'qaheuveetu.jpg', 'xmtolvgkaf.jpg']
测试文件名示例： ['vlvyvlvywu.jpg', 'xzhzduujjb.jpg', 'mwuenemtet.jpg', 'bkckzwkzde.jpg', 'doybvlwefm.jpg']
预测数量： 7065


In [13]:
# ========== 生成提交文件 ==========
submission = pd.DataFrame({
    "ID": test_ids,
    "Emotion": pred_labels   # 直接是 0~5，无需转换
})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(submission.head())
print("提交文件已生成：/kaggle/working/submission.csv")
print("标签分布：", Counter(pred_labels))

               ID  Emotion
0  tborkgneql.jpg        3
1  ocnyeeiise.jpg        0
2  xxzktmwpao.jpg        1
3  qaheuveetu.jpg        3
4  xmtolvgkaf.jpg        3
提交文件已生成：/kaggle/working/submission.csv
标签分布： Counter({np.int64(2): 1855, np.int64(5): 1264, np.int64(3): 1213, np.int64(0): 1001, np.int64(1): 968, np.int64(4): 764})


In [14]:
#消融实验
import gc
import time

# ---------- 全局设置 ----------
ABL_EPOCHS     = 6          # 快速看趋势；正式跑可调到 12~15
ABL_BATCH      = 32
ABL_NUM_WORKERS = 0         # Kaggle notebook 下建议 0，避免报错刷屏
ABL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("消融实验设备：", ABL_DEVICE)

# ---------- 数据增强工厂 ----------
def abl_train_tfms(img_size, strong=False):
    ops = [transforms.Resize((img_size, img_size))]
    if strong:
        ops += [
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
        ]
    else:
        ops += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
    if strong:
        ops.append(transforms.RandomErasing(p=0.25))
    return transforms.Compose(ops)


def abl_val_tfms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


# ---------- 模型工厂 ----------
def abl_build_model(name, num_classes, dropout=0.0):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_f = m.fc.in_features
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes)) if dropout > 0 \
               else nn.Linear(in_f, num_classes)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_f = m.fc.in_features
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes)) if dropout > 0 \
               else nn.Linear(in_f, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes)) if dropout > 0 \
                          else nn.Linear(in_f, num_classes)
    else:
        raise ValueError(f"未知模型: {name}")
    return m.to(ABL_DEVICE)


# ---------- 训练 / 验证 ----------
def abl_train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(ABL_DEVICE), labels.to(ABL_DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def abl_evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(ABL_DEVICE), labels.to(ABL_DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


# ---------- 单组实验 ----------
def abl_run_one(cfg):
    torch.manual_seed(42); np.random.seed(42)
    train_ds = FERDataset(X_tr, y_tr, transform=abl_train_tfms(cfg["img_size"], cfg["strong"]))
    val_ds   = FERDataset(X_val, y_val, transform=abl_val_tfms(cfg["img_size"]))

    if cfg["sampler"]:
        counts = Counter(y_tr)
        w = [1.0 / counts[y] for y in y_tr]
        sampler = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=ABL_BATCH, sampler=sampler,
                                  num_workers=ABL_NUM_WORKERS)
    else:
        train_loader = DataLoader(train_ds, batch_size=ABL_BATCH, shuffle=True,
                                  num_workers=ABL_NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=ABL_BATCH, shuffle=False,
                            num_workers=ABL_NUM_WORKERS)

    model = abl_build_model(cfg["model"], len(classes), dropout=cfg["dropout"])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ABL_EPOCHS)

    best_val = 0.0
    for ep in range(ABL_EPOCHS):
        tr_loss, tr_acc = abl_train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = abl_evaluate(model, val_loader, criterion)
        scheduler.step()
        best_val = max(best_val, val_acc)
        print(f"  [{cfg['name']}] ep{ep+1}/{ABL_EPOCHS} "
              f"tr_acc={tr_acc:.4f} tr_loss={tr_loss:.4f} "
              f"val_acc={val_acc:.4f} val_loss={val_loss:.4f}")

    del model, optimizer, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()
    return best_val


# ---------- 消融配置 ----------
# 想跑哪几组就留哪几组
ABL_CONFIGS = [
    # 1) 基线：完全复用你原来的设置（224 + resnet50 + 弱增强 + 无 dropout）
    dict(name="1_baseline_r50_224",
         model="resnet50", img_size=224, strong=False,
         dropout=0.0, wd=1e-4, lr=1e-4, sampler=True),

    # 2) 只换模型：resnet18
    dict(name="2_r18_224",
         model="resnet18", img_size=224, strong=False,
         dropout=0.0, wd=1e-4, lr=1e-4, sampler=True),

    # 3) 只换输入尺寸：64
    dict(name="3_r50_64",
         model="resnet50", img_size=64, strong=False,
         dropout=0.0, wd=1e-4, lr=1e-4, sampler=True),

    # 4) 只加强正则：dropout + 大 wd
    dict(name="4_r50_224_dropout_wd",
         model="resnet50", img_size=224, strong=False,
         dropout=0.5, wd=1e-2, lr=1e-4, sampler=True),

    # 5) 只加强增强
    dict(name="5_r50_224_strongaug",
         model="resnet50", img_size=224, strong=True,
         dropout=0.0, wd=1e-4, lr=1e-4, sampler=True),

    # 6) 去掉 sampler
    dict(name="6_r50_224_no_sampler",
         model="resnet50", img_size=224, strong=False,
         dropout=0.0, wd=1e-4, lr=1e-4, sampler=False),

    # 7) 全家桶：强增强 + dropout + 大 wd
    dict(name="7_r50_224_all",
         model="resnet50", img_size=224, strong=True,
         dropout=0.5, wd=1e-2, lr=1e-4, sampler=True),
]

# ---------- 依次执行 ----------
results = []
for cfg in ABL_CONFIGS:
    print("=" * 70)
    print(f"▶ 实验：{cfg['name']}")
    t0 = time.time()
    try:
        best_acc = abl_run_one(cfg)
        elapsed = time.time() - t0
        results.append({
            "实验": cfg["name"],
            "模型": cfg["model"],
            "IMG_SIZE": cfg["img_size"],
            "强增强": cfg["strong"],
            "Dropout": cfg["dropout"],
            "weight_decay": cfg["wd"],
            "Sampler": cfg["sampler"],
            "best_val_acc": round(best_acc, 4),
            "耗时(s)": round(elapsed, 1),
        })
    except Exception as e:
        print(f"  实验 {cfg['name']} 失败：{e}")
        results.append({
            "实验": cfg["name"], "模型": cfg["model"],
            "IMG_SIZE": cfg["img_size"], "强增强": cfg["strong"],
            "Dropout": cfg["dropout"], "weight_decay": cfg["wd"],
            "Sampler": cfg["sampler"],
            "best_val_acc": None, "耗时(s)": None,
        })

# ---------- 对照结果表 ----------
df_abl = pd.DataFrame(results).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
print("\n" + "=" * 70)
print("消融实验结果对照表（按 best_val_acc 降序）")
print(df_abl.to_string(index=False))

# 相对于基线的差值，便于一眼看出每个因素贡献多少
baseline = df_abl.loc[df_abl["实验"] == "1_baseline_r50_224", "best_val_acc"]
if len(baseline) > 0 and pd.notna(baseline.values[0]):
    base = baseline.values[0]
    df_abl["Δ vs 基线"] = (df_abl["best_val_acc"] - base).round(4)
    print("\n相对基线的差值（正=提升，负=下降）：")
    print(df_abl[["实验", "best_val_acc", "Δ vs 基线"]].to_string(index=False))

消融实验设备： cuda
▶ 实验：1_baseline_r50_224
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 110MB/s]


  [1_baseline_r50_224] ep1/6 tr_acc=0.5087 tr_loss=1.2414 val_acc=0.5915 val_loss=1.0979
  [1_baseline_r50_224] ep2/6 tr_acc=0.6493 tr_loss=0.9402 val_acc=0.6268 val_loss=0.9966
  [1_baseline_r50_224] ep3/6 tr_acc=0.7123 tr_loss=0.7856 val_acc=0.6480 val_loss=0.9678
  [1_baseline_r50_224] ep4/6 tr_acc=0.7763 tr_loss=0.6292 val_acc=0.6525 val_loss=0.9837
  [1_baseline_r50_224] ep5/6 tr_acc=0.8238 tr_loss=0.5068 val_acc=0.6620 val_loss=1.0009
  [1_baseline_r50_224] ep6/6 tr_acc=0.8586 tr_loss=0.4141 val_acc=0.6678 val_loss=1.0402
▶ 实验：2_r18_224
  [2_r18_224] ep1/6 tr_acc=0.5454 tr_loss=1.1686 val_acc=0.6056 val_loss=1.0251
  [2_r18_224] ep2/6 tr_acc=0.6431 tr_loss=0.9409 val_acc=0.6308 val_loss=1.0467
  [2_r18_224] ep3/6 tr_acc=0.7044 tr_loss=0.8099 val_acc=0.6334 val_loss=1.0195
  [2_r18_224] ep4/6 tr_acc=0.7586 tr_loss=0.6708 val_acc=0.6584 val_loss=1.0024
  [2_r18_224] ep5/6 tr_acc=0.8013 tr_loss=0.5639 val_acc=0.6754 val_loss=0.9649
  [2_r18_224] ep6/6 tr_acc=0.8362 tr_loss=0.4899 va